In [ ]:
import requests
import json
from os.path import expanduser
from requests.auth import HTTPBasicAuth

# load credentials
with open(expanduser('brain_credential_copy.txt')) as f:
    credentials = json.load(f)

# Extract username and password
username, password = credentials

# Create a session object
sess = requests.Session()

# Set up basic authentication
sess.auth = HTTPBasicAuth(username, password)

# Send a Post request to the API for authentication
response = sess.post('https://api.worldquantbrain.com/authentication')

# Print the response status and content for debugging
print(response.status_code)
print(response.json())

In [ ]:
r = sess.get(
    "https://api.worldquantbrain.com/data-fields",
    params={
        "instrumentType": "EQUITY",
        "region": "USA",
        "delay": 1,
        "universe": "TOP3000",
        "dataset.id": "news12",
        "limit": 1
    }
)

print(r.status_code)
print(r.json())


获取数据集ID为fundamental6下的所有数据字段

In [ ]:
def get_datafields(
    s,
    searchScope,
    dataset_id: str = '',
    search: str = ''
):
    import pandas as pd
    import time

    instrument_type = searchScope['instrumentType']
    region = searchScope['region']
    delay = searchScope['delay']
    universe = searchScope['universe']

    base_url = "https://api.worldquantbrain.com/data-fields"

    params = {
        "instrumentType": instrument_type,
        "region": region,
        "delay": delay,
        "universe": universe,
        "limit": 50,
        "offset": 0
    }

    if dataset_id:
        params["dataset.id"] = dataset_id
    if search:
        params["search"] = search

    def safe_get(params, max_retry=5):
        sleep_time = 1.0
        for i in range(max_retry):
            r = s.get(base_url, params=params)
            j = r.json()

            # 正常返回
            if "results" in j:
                return j

            # Rate limit
            if "rate limit" in str(j).lower():
                time.sleep(sleep_time)
                sleep_time *= 2   # ⭐ 指数退避
                continue

            # 其他错误直接抛
            raise RuntimeError(j)

        raise RuntimeError("Exceeded max retries due to rate limit")

    # 第一次请求
    j = safe_get(params)
    count = j.get("count", len(j["results"]))

    all_results = []
    all_results.extend(j["results"])

    # 分页
    for offset in range(50, count, 50):
        params["offset"] = offset
        j = safe_get(params)
        all_results.extend(j["results"])
        time.sleep(0.3)  # ⭐ 比 0.2 更稳

    return pd.DataFrame(all_results)


In [ ]:
    
searchScope={'region':'USA','delay':'1','universe':'TOP3000','instrumentType':'EQUITY'}

fundamental6=get_datafields(s=sess,searchScope=searchScope,dataset_id='fundamental6')


In [ ]:
fundamental6=fundamental6[fundamental6['type']=="MATRIX"]
len(fundamental6)


In [ ]:
datafields_list_fundamental6=fundamental6['id'].values

In [ ]:
datafields_list_fundamental6

In [ ]:
searchScope={'region':'USA','delay':'1','universe':'TOP3000','instrumentType':'EQUITY'}

news12=get_datafields(s=sess,searchScope=searchScope,dataset_id='news12')


In [ ]:
news12=news12[news12['type']=="MATRIX"]
len(news12)


In [ ]:
datafields_list_news12=news12['id'].values
datafields_list_news12

In [ ]:
alpha_list=[]

for datafield in datafields_list_fundamental6:
#替换新的测试字段
#for datafield in datafields_list_news12:
    print("正在将下面alpha表达式与setting封装")
    alpha_expression=f'group_rank({datafield}/invested_capital,industry)'
    print(alpha_expression)
    simulation_data={
    'type':'REGULAR',
    'settings':{
        'instrumentType':'EQUITY',
        'region':'USA',
        'universe':'TOP3000',
        'delay':1,
        'decay':1,#设置为60天的衰减
        'neutralization':'SUBINDUSTRY',
        'truncation':0.08,
        'pasteurization':'ON',
        'unitHandling':'VERIFY',
        'nanHandling':'ON',
        'language':'FASTEXPR',
        'visualization':False,
            },
    'regular':alpha_expression
    }

    alpha_list.append(simulation_data)

print(f'there are {len(alpha_list)} Alphas to simulate')



In [ ]:
alpha_list[2]

将alpha一个个回测

In [ ]:
import time
import json
import os
from datetime import datetime
import traceback

CHECKPOINT_PATH = "alpha_checkpoint.json"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        try:
            with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            # 读取失败就重建空字典
            return {}
    return {}

def save_checkpoint(cp):
    tmp = CHECKPOINT_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(cp, f, ensure_ascii=False, indent=2)
    os.replace(tmp, CHECKPOINT_PATH)

def now_iso():
    return datetime.utcnow().isoformat() + "Z"

def get_retry_after_seconds(resp):
    # 保持你原来行为：没有 header 或解析失败视为 '0'
    val = resp.headers.get("Retry-After", "0")
    try:
        return float(val)
    except Exception:
        return 0.0

# --- main loop (替换原有 for alpha in alpha_list: ...) ---
checkpoint = load_checkpoint()

for idx, alpha in enumerate(alpha_list):
    key = str(idx)  # 用列表索引作为 checkpoint key，简单且稳定
    entry = checkpoint.get(key, {})

    # 如果已经完成就跳过
    if entry.get("status") == "completed":
        print(f"[{idx}] already completed (alpha_id={entry.get('alpha_id')}) — skip")
        continue

    # 如果之前有提交记录且有进度 URL，则可以从那里继续
    sim_progress_url = entry.get("sim_progress_url")
    if not sim_progress_url:
        # 还没提交过，尝试提交
        try:
            print(f"[{idx}] submitting alpha...")
            sim_resp = sess.post("https://api.worldquantbrain.com/simulations", json=alpha)
            # 尝试读取 Location header
            sim_progress_url = sim_resp.headers.get("Location")
            if not sim_progress_url:
                # 没有 Location：记录并等待下一轮重试（和你原来行为一致）
                print(f"[{idx}] no Location header after submit — will sleep 10s and continue to next alpha")
                checkpoint[key] = {
                    "status": "submit_failed",
                    "last_submit_resp_status": sim_resp.status_code,
                    "last_submit_time": now_iso()
                }
                save_checkpoint(checkpoint)
                time.sleep(10)
                continue
            # 有 Location，记录并进入轮询
            checkpoint[key] = {
                "status": "submitted",
                "sim_progress_url": sim_progress_url,
                "submitted_time": now_iso()
            }
            save_checkpoint(checkpoint)
            print(f"[{idx}] submitted, progress URL: {sim_progress_url}")
        except Exception as e:
            print(f"[{idx}] submit exception: {e}")
            traceback.print_exc()
            checkpoint[key] = {
                "status": "submit_exception",
                "last_error": str(e),
                "last_submit_time": now_iso()
            }
            save_checkpoint(checkpoint)
            # 按你原来逻辑，等待 10 秒再去下一个 alpha
            time.sleep(10)
            continue

    # 开始/继续轮询进度
    last_log_time = time.time()
    # 保证一开始也写一次进度到 checkpoint，以便重启后继续
    checkpoint.setdefault(key, {}).update({
        "status": "submitted",
        "sim_progress_url": sim_progress_url,
        "last_polled_time": now_iso()
    })
    save_checkpoint(checkpoint)

    print(f"[{idx}] start polling {sim_progress_url}")
    while True:
        try:
            sim_progress_resp = sess.get(sim_progress_url)
            retry_after_sec = get_retry_after_seconds(sim_progress_resp)

            # 尝试解析返回 body（如果是 json）
            body = {}
            try:
                body = sim_progress_resp.json()
            except Exception:
                # 解析失败就空字典
                body = {}

            # 每隔 10 分钟写一次 checkpoint（记录最新返回的 body）
            now_t = time.time()
            if now_t - last_log_time >= 600:
                checkpoint[key].update({
                    "last_progress": body,
                    "last_polled_time": now_iso()
                })
                save_checkpoint(checkpoint)
                print(f"[{idx}] logged progress at {now_iso()} (every-10min)")
                last_log_time = now_t

            # 如果 Retry-After == 0（和你原版判断一致），视为结束
            if retry_after_sec == 0:
                alpha_id = body.get("alpha") or body.get("id")
                checkpoint[key].update({
                    "status": "completed",
                    "alpha_id": alpha_id,
                    "completed_time": now_iso(),
                    "last_response": body
                })
                save_checkpoint(checkpoint)
                print(f"[{idx}] completed, alpha_id={alpha_id}")
                break

            # 否则按服务器要求等待
            # 但防止服务器给出过长值导致长时间不记录：我们仍然每 600s 写一次（上面的逻辑）
            sleep_for = retry_after_sec
            # safety: 最少等待 1s，最多等待 600s（10min）——避免被服务器返回超长导致长时间没有写盘
            if sleep_for < 1:
                sleep_for = 1
            if sleep_for > 600:
                sleep_for = 600
            time.sleep(sleep_for)

        except Exception as e:
            # 轮询过程中出错：保存状态以便下次恢复，然后跳出循环去处理下一个 alpha
            print(f"[{idx}] poll exception: {e} — saving state and moving to next alpha")
            traceback.print_exc()
            checkpoint[key].update({
                "status": "submitted",
                "last_error": str(e),
                "last_polled_time": now_iso()
            })
            save_checkpoint(checkpoint)
            # 等短时间再去下一个 alpha（按原逻辑）
            time.sleep(10)
            break
